---
## 0 — Installation & Imports

In [ ]:
!pip install scikit-learn matplotlib seaborn scipy xgboost tqdm optuna shap --quiet
!pip install lightgbm --config-settings=cmake.define.USE_GPU=ON --quiet 2>/dev/null || \
    pip install lightgbm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings, subprocess, os
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, brier_score_loss,
    log_loss, classification_report, RocCurveDisplay,
    PrecisionRecallDisplay, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import lightgbm as lgb

np.random.seed(42)

---
## 1 — Configuration

In [ ]:
DATA_PATH    = "/content/drive/MyDrive/collab/f2/india+portugal+historical+population+f.csv"
OUTPUT_DIR   = "/content/drive/MyDrive/collab/f2/v10/"

TARGET_COL   = "in_fire"
RANDOM_STATE = 42

# ── Grille spatiale (Portugal entier) ────────────────────────
N_BLOCKS_X, N_BLOCKS_Y = 4, 4   # 25 blocs

# ── Nested Cross-Validation spatiale ─────────────────────────
K_TEST_BLOCKS    = 3     # blocs TEST par iteration
N_CV_ITERATIONS  = 5     # iterations (chacune a son propre Optuna)

# ── Optuna (1 par iteration) ─────────────────────────────────
N_OPTUNA_TRIALS  = 10    # trials par iteration (peu car x8 iterations)
N_OPTUNA_SPLITS  = 2     # splits par trial
K_VAL_BLOCKS     = 3     # blocs val par split

# ── Divers ───────────────────────────────────────────────────
MIN_FIRES_PER_BLOCK = 0
MAX_FIRE_RATE_TRAIN = 1
RADIUS_KM = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Nested CV: {K_TEST_BLOCKS} blocs test x {N_CV_ITERATIONS} iterations")
print(f"Optuna: {N_OPTUNA_TRIALS} trials x {N_OPTUNA_SPLITS} splits PAR ITERATION")
print(f"Total Optuna fits: ~{N_CV_ITERATIONS * N_OPTUNA_TRIALS * N_OPTUNA_SPLITS}")
print(f"Grille: {N_BLOCKS_X}x{N_BLOCKS_Y} = {N_BLOCKS_X*N_BLOCKS_Y} blocs")

Nested CV: 3 blocs test x 5 iterations
Optuna: 10 trials x 2 splits PAR ITERATION
Total Optuna fits: ~100
Grille: 4x4 = 16 blocs


---
## 2 — Data Loading & Features

In [ ]:

!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
data = pd.read_csv(DATA_PATH)
print(f"Shape: {data.shape} | Fire rate: {data[TARGET_COL].mean():.2%}")
hist_cols = [c for c in data.columns if 'was_in_fire' in c]
print(f"Colonnes historiques: {hist_cols}")

Shape: (89198, 56) | Fire rate: 9.94%
Colonnes historiques: ['was_in_fire_2021', 'was_in_fire_2021_distance', 'was_in_fire_2022', 'was_in_fire_2022_distance', 'was_in_fire_2023', 'was_in_fire_2023_distance']


In [ ]:
# Encode categoricals (user v5)
for col in ['TEXTURE_SOTER']:
    if col in data.columns and data[col].dtype == object:
        data[col] = LabelEncoder().fit_transform(data[col].astype(str))

In [ ]:

# === v9 NEW: Historical fire distance features ===
HIST_FIRE_COLS = ['was_in_fire_2021', 'was_in_fire_2022', 'was_in_fire_2023']
HIST_DIST_COLS = ['was_in_fire_2021_distance', 'was_in_fire_2022_distance', 'was_in_fire_2023_distance']
hist_present = [c for c in HIST_FIRE_COLS if c in data.columns]
dist_present = [c for c in HIST_DIST_COLS if c in data.columns]
print(f"Feux historiques: {hist_present}")
print(f"Distances historiques: {dist_present}")



# Features de distance historique
if len(dist_present) > 0:
    data['min_hist_fire_dist'] = data[dist_present].min(axis=1)
    data['mean_hist_fire_dist'] = data[dist_present].mean(axis=1)




Feux historiques: ['was_in_fire_2021', 'was_in_fire_2022', 'was_in_fire_2023']
Distances historiques: ['was_in_fire_2021_distance', 'was_in_fire_2022_distance', 'was_in_fire_2023_distance']


### 2.2 — Voisinage spatial

In [ ]:
from scipy.spatial import cKDTree
import time



# ══════════════════════════════════════════════════════════════
# FIRE DENSITY: utilise les feux HISTORIQUES (2021-2023)
# Pas de leakage car le target (in_fire) est une annee differente
# ══════════════════════════════════════════════════════════════
HIST_FIRE_COLS = ['was_in_fire_2021', 'was_in_fire_2022', 'was_in_fire_2023']
hist_cols_present = [c for c in HIST_FIRE_COLS if c in data.columns]
print(f"Colonnes historiques trouvees: {hist_cols_present}")


start = time.time()

# Coordonnees en km
mean_lat = np.deg2rad(data['latitude'].mean())
x_km = data['longitude'].values * 111.0 * np.cos(mean_lat)
y_km = data['latitude'].values * 111.0
coords = np.column_stack([x_km, y_km])

tree = cKDTree(coords)
neighbors_list = tree.query_ball_tree(tree, r=RADIUS_KM)



# Feux historiques combines (moyenne 2021-2023)
if len(hist_cols_present) > 0:
    hist_fire = data[hist_cols_present].mean(axis=1).values  # proba historique
else:
    hist_fire = None

n = len(data)
neigh_n_points = np.zeros(n)
neigh_hist_fire_density = np.zeros(n)  # densite feux HISTORIQUES
neigh_forest_pct = np.zeros(n)

for i in range(n):
    nb_idx = neighbors_list[i]
    neigh_n_points[i] = len(nb_idx)
    if len(nb_idx) < 2: continue



    if hist_fire is not None:
        neigh_hist_fire_density[i] = np.mean(hist_fire[nb_idx])


data['neigh_n_points'] = neigh_n_points
data['neigh_forest_pct'] = neigh_forest_pct
data['neigh_hist_fire_density'] = neigh_hist_fire_density

elapsed = time.time() - start
print(f"\nDone en {elapsed:.0f}s — {n:,} points, "
      f"~{np.mean([len(x) for x in neighbors_list]):.0f} voisins/point")
if hist_fire is not None:
    print(f"neigh_hist_fire_density: mean={data['neigh_hist_fire_density'].mean():.3f}, "
          f"max={data['neigh_hist_fire_density'].max():.3f}")

Colonnes historiques trouvees: ['was_in_fire_2021', 'was_in_fire_2022', 'was_in_fire_2023']

Done en 2s — 89,198 points, ~75 voisins/point
neigh_hist_fire_density: mean=0.095, max=0.944


### 2.3 — Configuration des features (modifier ici)

In [ ]:
# ══════════════════════════════════════════════════════════════
# FEATURE CONFIGURATION — modifier ici (True/False ou 1/0)
# ══════════════════════════════════════════════════════════════

FEATURE_GROUPS = {
    # === Sol ===
    'sol_base':          1,   # COARSE, SAND, SILT, CLAY, BULK, ORG_CARBON, PH_WATER
    'sol_cec':           1,   # CEC_SOIL, BSAT, ALUM_SAT, ESP
    'sol_autres':        1,   # GYPSUM, ELEC_COND, CN_RATIO
    'Foret': 1,
    # === Climat ===
    'climat_base':       1,   # prec_ete, tmax_ete, tmax_hiver, tmin_ete
    'climat_diff':       1,   # dif_perc_*, dif_max_*, dif_min_*

    # === Topographie ===
    'topo':              1,   # Elevation
    'population':         1,
    # === Remote sensing ===
    'remote':            1,   # landcover

    # === Historique feux binaire ===
    'hist_fire_binary':  0,   # was_in_fire_2021/2022/2023,

    # === Historique feux distances ===
    'hist_fire_dist':    1,   # was_in_fire_xxxx_distance, min/mean_hist_fire_dist,


    # === Voisinage feux historiques ===
    'voisinage_fire':    1,   # neigh_hist_fire_density
}

# Mapping groupes -> colonnes
GROUP_COLS = {
    'population':    ['population'],
    'Foret' : ['is_Eucalipto','is_Outras_folhosas','is_Pinheiro_bravo','is_Sobreiro'],
    'sol_base':      ['COARSE','SAND','SILT','CLAY','BULK','ORG_CARBON','PH_WATER'],
    'sol_cec':       ['CEC_SOIL','BSAT','ALUM_SAT'],
    'sol_autres':    ['GYPSUM','ELEC_COND','CN_RATIO'],
    'climat_base':   ['prec_ete','tmax_ete','tmax_hiver','tmin_ete'],
    'climat_diff':   [c for c in data.columns if c.startswith('dif_')],
    'topo':          ['Elevation'],
    'remote':        ['landcover'],
    'hist_fire_binary': ['was_in_fire_2021','was_in_fire_2022','was_in_fire_2023'],
    'hist_fire_dist': ['was_in_fire_2021_distance','was_in_fire_2022_distance',
                       'was_in_fire_2023_distance','min_hist_fire_dist',
                       'mean_hist_fire_dist'],
    'voisinage_fire': ['neigh_hist_fire_density'],
}

# Construire FEATURE_COLS
ALWAYS_EXCLUDE = {TARGET_COL, 'latitude', 'longitude', 'band',
                  'block_x', 'block_y', 'block_id', 'TEXTURE_SOTER',
                  'TEXTURE_USDA', 'CEC_CLAY', 'CEC_EFF', 'TCARBON_EQ'}

active = []
for g, on in FEATURE_GROUPS.items():
    if on: active.extend([c for c in GROUP_COLS.get(g,[]) if c in data.columns])
FEATURE_COLS = list(dict.fromkeys(c for c in active if c not in ALWAYS_EXCLUDE))

assert 'latitude' not in FEATURE_COLS and 'longitude' not in FEATURE_COLS
assert TARGET_COL not in FEATURE_COLS
for c in FEATURE_COLS:
    assert 'in_fire' not in c or 'was_in_fire' in c, f"LEAKAGE: {c}"

print(f"Features: {len(FEATURE_COLS)}")
print(f"Actifs: {[g for g,e in FEATURE_GROUPS.items() if e]}")
print(f"Desactives: {[g for g,e in FEATURE_GROUPS.items() if not e]}")
print(f"\n{FEATURE_COLS}")

Features: 38
Actifs: ['sol_base', 'sol_cec', 'sol_autres', 'Foret', 'climat_base', 'climat_diff', 'topo', 'population', 'remote', 'hist_fire_dist', 'voisinage_fire']
Desactives: ['hist_fire_binary']

['COARSE', 'SAND', 'SILT', 'CLAY', 'BULK', 'ORG_CARBON', 'PH_WATER', 'CEC_SOIL', 'BSAT', 'ALUM_SAT', 'GYPSUM', 'ELEC_COND', 'CN_RATIO', 'is_Eucalipto', 'is_Outras_folhosas', 'is_Pinheiro_bravo', 'is_Sobreiro', 'prec_ete', 'tmax_ete', 'tmax_hiver', 'tmin_ete', 'dif_perc_hiver_printemps', 'dif_perc_automne_printemps', 'dif_perc_ete_automne', 'dif_max_automne_printemps', 'dif_max_hiver_printemps', 'dif_min_hiver_printemps', 'dif_min_automne_hiver', 'dif_max_hiver_min_automne', 'Elevation', 'population', 'landcover', 'was_in_fire_2021_distance', 'was_in_fire_2022_distance', 'was_in_fire_2023_distance', 'min_hist_fire_dist', 'mean_hist_fire_dist', 'neigh_hist_fire_density']
